# 🗽 BÁO CÁO ĐỒ ÁN CUỐI KỲ: TRỰC QUAN HÓA DỮ LIỆU & PHÂN TÍCH THỊ TRƯỜNG BẤT ĐỘNG SẢN NEW YORK (NYC)
---
### 🎯 CÁC YÊU CẦU THỰC HIỆN:
1. **Thu thập & Tiền xử lý dữ liệu:** Dữ liệu thô > 5,000 dòng từ Kaggle, kết nối 3 bảng qua khóa ZIP CODE, làm sạch missing values/outliers, tạo các trường tính toán mới.
2. **Khám phá dữ liệu (EDA):** Dùng Matplotlib & Seaborn vẽ các biểu đồ tĩnh cơ bản phân tích phân phối, tương quan.
3. **Dashboard tương tác (Streamlit + Plotly):** Bộ lọc đa chiều, drill-down phân cấp, tích hợp 11 loại biểu đồ khác nhau bao gồm bản đồ địa lý.
4. **Insight & Dự báo:** Xây dựng mô hình Hồi quy tuyến tính (Linear Regression) đa biến, phân tích hệ số tác động và dự báo xu hướng giá BĐS.


## 📁 PHẦN 1: THU THẬP & TIỀN XỬ LÝ DỮ LIỆU (YÊU CẦU 1)
- Nguồn dữ liệu chính: `nyc-rolling-sales.csv` (>84,000 dòng từ Kaggle).
- Dữ liệu liên kết: `NYC Zipcodes - NYC Zip 2020.csv` và `nyc_demographics.csv`.
- Các bước tiền xử lý: Ép kiểu dữ liệu số, loại bỏ NaN, lọc Outliers phi thực tế, Join 3 bảng và Feature Engineering.

In [ ]:
import pandas as pd
import numpy as np

# BƯỚC 1: ĐỌC DỮ LIỆU TỪ 3 BẢNG (Thu thập)
print("Đang tải dữ liệu thô...")
df_sales = pd.read_csv("nyc-rolling-sales.csv")
df_zip = pd.read_csv("NYC Zipcodes - NYC Zip 2020.csv")
df_demo = pd.read_csv("nyc_demographics.csv")

# BƯỚC 2: TIỀN XỬ LÝ & LÀM SẠCH (Cleaning)
# 2.1 Ép kiểu dữ liệu số
df_sales["SALE PRICE"] = pd.to_numeric(df_sales["SALE PRICE"], errors="coerce")
df_sales["GROSS SQUARE FEET"] = pd.to_numeric(df_sales["GROSS SQUARE FEET"], errors="coerce")
df_sales["YEAR BUILT"] = pd.to_numeric(df_sales["YEAR BUILT"], errors="coerce")

# 2.2 Khử missing values
df_sales = df_sales.dropna(subset=["SALE PRICE", "GROSS SQUARE FEET", "YEAR BUILT", "ZIP CODE"])

# 2.3 Khử ngoại lai (Outliers)
# Loại bỏ giao dịch tặng/chuyển nhượng nội bộ (< $10,000) và diện tích <= 0
df_sales = df_sales[(df_sales["SALE PRICE"] >= 10000) & 
                    (df_sales["GROSS SQUARE FEET"] > 0) & 
                    (df_sales["YEAR BUILT"] > 1800)]

# BƯỚC 3: KẾT NỐI NHIỀU BẢNG (Merge/Join)
df_zip.rename(columns={"ZIP Codes": "ZIP CODE"}, inplace=True)
df_merged = pd.merge(df_sales, df_zip, on="ZIP CODE", how="inner")
df_final = pd.merge(df_merged, df_demo, on="ZIP CODE", how="left")

# BƯỚC 4: TẠO CÁC TRƯỜNG DỮ LIỆU TÍNH TOÁN (Calculated Fields)
df_final["SALE DATE"] = pd.to_datetime(df_final["SALE DATE"])
df_final["SALE_YEAR"] = df_final["SALE DATE"].dt.year
df_final["SALE_MONTH"] = df_final["SALE DATE"].dt.month
df_final["SALE_YEAR_MONTH"] = df_final["SALE DATE"].dt.to_period("M").astype(str)

# Tuổi thọ công trình lúc bán
df_final["PROPERTY_AGE"] = df_final["SALE_YEAR"] - df_final["YEAR BUILT"]
df_final = df_final[df_final["PROPERTY_AGE"] >= 0]

# Đơn giá trên mỗi đơn vị diện tích (USD / sqft)
df_final["PRICE_PER_SQFT"] = df_final["SALE PRICE"] / df_final["GROSS SQUARE FEET"]

# Logarit hóa giá bán để chuẩn hóa phân phối hồi quy
df_final["LOG_SALE_PRICE"] = np.log1p(df_final["SALE PRICE"])

# Phân khúc giá nhà (3 nhóm quantile)
df_final["PRICE_SEGMENT"] = pd.qcut(df_final["SALE PRICE"], q=3, labels=["Phổ thông", "Tầm trung", "Cao cấp"])

# Đồng bộ tên cột tiếng Việt cho Dashboard
df_final["QUAN"] = df_final["Borough"]
df_final["KHU_PHO"] = df_final["Neighborhood"]
df_final["DIEN_TICH_SU_DUNG"] = df_final["GROSS SQUARE FEET"]
df_final["DIEN_TICH_DAT"] = df_final["LAND SQUARE FEET"]
df_final["GIA_BAN"] = df_final["SALE PRICE"]
df_final["GIA_TREN_SQFT"] = df_final["PRICE_PER_SQFT"]
df_final["TUOI_THO_NHA"] = df_final["PROPERTY_AGE"]
df_final["THU_NHAP_TRUNG_BINH"] = df_final["MEDIAN_HOUSEHOLD_INCOME"]
df_final["THANG_NAM_BAN"] = df_final["SALE_YEAR_MONTH"]
df_final["LOG_GIA_BAN"] = df_final["LOG_SALE_PRICE"]
df_final["PHAN_KHUC_GIA"] = df_final["PRICE_SEGMENT"]

# Lưu file dữ liệu sạch hoàn thiện
df_final.to_csv("final_dashboard_data.csv", index=False)
df_final.to_csv("du_lieu_da_xu_ly.csv", index=False)
df_final.to_csv("du_lieu_dashboard_tieng_viet.csv", index=False)
print(f"✅ Hoàn thành tiền xử lý! Tập dữ liệu sạch có: {df_final.shape[0]} dòng và {df_final.shape[1]} cột.")


## 📊 PHẦN 2: KHÁM PHÁ DỮ LIỆU & PHÂN TÍCH PHÂN PHỐI - EDA (YÊU CẦU 2)
Sử dụng thư viện **Matplotlib** và **Seaborn** để trực quan hóa phân phối biến số, phát hiện tương quan và biên độ giá.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_eda = pd.read_csv("final_dashboard_data.csv", low_memory=False)
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram: Phân phối Log(Giá Bán)
sns.histplot(df_eda["LOG_SALE_PRICE"], bins=50, kde=True, ax=axes[0, 0], color="royalblue")
axes[0, 0].set_title("1. Phân Phối Logarit Giá Bán Nhà (Log Sale Price)", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Log Sale Price (USD)")
axes[0, 0].set_ylabel("Số lượng giao dịch")

# 2. Boxplot: Phân phối giá giữa 5 Quận
sns.boxplot(x="Borough", y="LOG_SALE_PRICE", data=df_eda, ax=axes[0, 1], palette="Set2")
axes[0, 1].set_title("2. Mức Giá Nhà Phân Hóa Giữa Các Quận (Borough)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Quận (Borough)")
axes[0, 1].set_ylabel("Log Sale Price (USD)")

# 3. Scatter Plot: Diện tích vs Giá bán
sns.scatterplot(x="GROSS SQUARE FEET", y="SALE PRICE", data=df_eda, alpha=0.35, ax=axes[1, 0], color="seagreen")
axes[1, 0].set_title("3. Mối Tương Quan Giữa Diện Tích và Giá Bán", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Diện tích (Sqft)")
axes[1, 0].set_ylabel("Giá Bán (USD)")

# 4. Heatmap: Ma trận tương quan
num_cols = ["SALE PRICE", "GROSS SQUARE FEET", "PROPERTY_AGE", "MEDIAN_HOUSEHOLD_INCOME"]
corr = df_eda[num_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=axes[1, 1], square=True, linewidths=.5)
axes[1, 1].set_title("4. Ma Trận Tương Quan Các Biến Số Học", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


### 📌 INSIGHT CHÍNH RÚT RA TỪ EDA:
1. **Phân phối giá nhà:** Sau khi lấy Logarit, phân phối giá nhà tiệm cận phân phối chuẩn (Normal Distribution), chứng minh việc dùng logarit giá bán là cơ sở khoa học để xây dựng mô hình hồi quy.
2. **Phân hóa địa lý:** Manhattan có trung vị giá cao nhất và độ phân tán lớn nhất, kế đến là Brooklyn. Bronx và Staten Island có mặt bằng giá thấp hơn.
3. **Tương quan:** Diện tích sử dụng và Thu nhập hộ gia đình tương quan thuận mạnh mẽ với Giá bán, đóng vai trò biến giải thích cốt lõi trong mô hình dự báo.

## 🤖 PHẦN 3: HỌC MÁY & HỒI QUY DỰ BÁO XU HƯỚNG (YÊU CẦU 4)
Xây dựng mô hình **Hồi quy tuyến tính đa biến (Multiple Linear Regression)** nhằm:
- Dự báo giá trị nhà dựa trên: Diện tích (`GROSS SQUARE FEET`), Tuổi thọ (`PROPERTY_AGE`), Thu nhập (`MEDIAN_HOUSEHOLD_INCOME`) và Quận (`Borough` One-Hot Encoded).
- Đánh giá chất lượng mô hình thông qua R2, MAE, RMSE.
- Cập nhật cột `GIA_DU_BAO` và `PREDICTED_PRICE` cho Dashboard.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

df_ml = pd.read_csv("final_dashboard_data.csv", low_memory=False)

# 1. Chuẩn bị biến X và y
X_num = df_ml[["GROSS SQUARE FEET", "PROPERTY_AGE", "MEDIAN_HOUSEHOLD_INCOME"]]
borough_dummies = pd.get_dummies(df_ml["Borough"], drop_first=True, dtype=float)
X = pd.concat([X_num, borough_dummies], axis=1)
y = df_ml["LOG_SALE_PRICE"]

# 2. Chia tập Train / Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Huấn luyện mô hình Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

# 4. Đánh giá chất lượng mô hình trên tập Test
y_pred_test = model.predict(X_test)
r2 = r2_score(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
rmse = root_mean_squared_error(y_test, y_pred_test)

print(f"🎯 Đánh giá mô hình Hồi quy tuyến tính:")
print(f"  - R2 Score: {r2:.4f} ({r2*100:.2f}%)")
print(f"  - MAE (Log): {mae:.4f}")
print(f"  - RMSE (Log): {rmse:.4f}")

# 5. Ghi nhận dự báo toàn tập dữ liệu
pred_log_all = model.predict(X)
df_ml["PREDICTED_LOG_PRICE"] = pred_log_all
df_ml["PREDICTED_PRICE"] = np.expm1(pred_log_all)
df_ml["LOG_GIA_DU_BAO"] = pred_log_all
df_ml["GIA_DU_BAO"] = np.expm1(pred_log_all)

# Lưu lại cập nhật cho cả 3 file dữ liệu
df_ml.to_csv("final_dashboard_data.csv", index=False)
df_ml.to_csv("du_lieu_da_xu_ly.csv", index=False)
df_ml.to_csv("du_lieu_dashboard_tieng_viet.csv", index=False)
print("✅ Đã cập nhật thành công các cột giá dự báo GIA_DU_BAO và PREDICTED_PRICE!")


## 🚀 PHẦN 4: KHỞI CHẠY DASHBOARD TRỰC QUAN HÓA (YÊU CẦU 3 & 4)
Dashboard đã được phát triển hoàn thiện trong file `app.py` với **11 loại biểu đồ** khác nhau, hỗ trợ bộ lọc và drill-down:
```bash
streamlit run app.py
```
Truy cập tại địa chỉ: `http://localhost:8501` để xem giao diện trực quan và trải nghiệm bộ công cụ dự báo giá AI tương tác.